# Touster — one-notebook LoRA fine-tuning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Patan-Sameer66/touster/blob/main/touster.ipynb)

**Fine-tune a small LLM end-to-end — dataset → self-improving hyperparameter search → compare → export — from one notebook.** No ML boilerplate. Runs on a free Colab **T4 GPU** or locally on **CPU**.

---

### 🚀 Quickstart
1. **Runtime → Change runtime type → T4 GPU** (free), then **Runtime → Run all**.
2. Edit **Cell 2 (Config)** — base model + dataset mode. That's the only cell you normally touch.
3. Watch the self-improvement loop find the best recipe, then collect your adapter / GGUF at the bottom.

###  Three ways to get a dataset (`DATASET_MODE` in Cell 2)
| Mode | You provide | Touster does |
|------|-------------|--------------|
| **0 · generate** | a topic string | LLM writes Q&A pairs for you |
| **1 · structure** | raw `.txt` / `.md` | LLM turns it into Q&A pairs |
| **2 · bring-your-own** | `.jsonl`, a URL, or a 🤗 HF dataset id | loads & converts it directly |

> 💡 An LLM (local **Ollama** or any OpenAI-compatible API) is only needed for modes 0/1 and the optional LLM-judge. Leave it blank and everything still runs with a zero-cost heuristic loop.

*A one-line note above each cell explains what it does — you rarely need to edit anything below Cell 2.*

### 1 · Hardware check
Detects your GPU / CPU / RAM and prints a table of models that fit, with rough speed estimates. Note a model name for Cell 2. Runs a tiny install — nothing to configure.

In [1]:
import sys, os, subprocess
from pathlib import Path

# ── Anchor cwd in case kernel's working directory was deleted ──────────
try:
    os.getcwd()
except OSError:
    os.chdir(Path.home())
_safe_cwd = str(Path.home())

# ── Detect environment ─────────────────────────────────────────────────
IN_COLAB   = 'google.colab' in sys.modules
ON_WINDOWS = sys.platform == 'win32'
ON_MAC     = sys.platform == 'darwin'
ON_LINUX   = sys.platform.startswith('linux')

env_name = 'Colab' if IN_COLAB else 'Windows' if ON_WINDOWS else 'macOS' if ON_MAC else 'Linux'
print(f'Environment : {env_name}')
print(f'Python      : {sys.version.split()[0]}')

# ── Light install for hardware detection ───────────────────────────────
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'psutil', 'nvidia-ml-py', '-q'],
    capture_output=True, cwd=_safe_cwd,
)

import psutil

# ── GPU detect ─────────────────────────────────────────────────────────
PLATFORM = 'cpu'
gpu_name  = None
vram_gb   = 0
bw_gbps   = 0

try:
    import pynvml
    pynvml.nvmlInit()
    h        = pynvml.nvmlDeviceGetHandleByIndex(0)
    gpu_name = pynvml.nvmlDeviceGetName(h)
    vram_gb  = pynvml.nvmlDeviceGetMemoryInfo(h).total / 1024**3
    PLATFORM = 'cuda'
    BW = {
        'T4': 300, 'A10': 600, 'A100 40': 1555, 'A100 80': 2039,
        'V100': 900, 'L4': 300, 'RTX 4090': 1008, 'RTX 3090': 936,
        'RTX 4080': 717, 'RTX 4070': 504, 'RTX 3080': 760,
    }
    bw_gbps = next((v for k, v in BW.items() if k.lower() in gpu_name.lower()), 250)
    pynvml.nvmlShutdown()
except Exception:
    pass

if PLATFORM == 'cpu' and ON_MAC:
    try:
        import mlx  # noqa: F401
        PLATFORM = 'mlx'
        gpu_name = 'Apple Silicon (MLX)'
        bw_gbps  = 200
    except ImportError:
        pass

ram_gb = psutil.virtual_memory().total / 1024**3

print(f'GPU         : {gpu_name or "none"}')
print(f'VRAM        : {vram_gb:.1f} GB' if vram_gb else 'VRAM        : n/a')
print(f'RAM         : {ram_gb:.1f} GB')
print(f'Backend     : {PLATFORM}')
print()

MODELS = [
    ('unsloth/Qwen2.5-0.5B-Instruct',      0.5,  0.5),
    ('unsloth/Llama-3.2-1B-Instruct',       1.0,  1.5),
    ('unsloth/Qwen2.5-3B-Instruct',         3.0,  4.0),
    ('unsloth/Llama-3.2-3B-Instruct',       3.0,  3.5),
    ('unsloth/Phi-3.5-mini-instruct',        3.8,  5.0),
    ('unsloth/gemma-2-2b-it',               2.0,  4.5),
    ('unsloth/Qwen2.5-7B-Instruct',         7.0,  6.0),
    ('unsloth/mistral-7b-instruct-v0.3',    7.0,  6.5),
    ('unsloth/Meta-Llama-3.1-8B',           8.0,  7.0),
    ('unsloth/gemma-2-9b-it',               9.0,  7.5),
]

print(f'{"#":<3} {"model":42} {"params":>7} {"vram(4bit)":>11} {"fit":>5} {"t/s":>6}')
print('-' * 76)
first_fit = True
for i, (mid, params, vram_need) in enumerate(MODELS):
    tps  = int(bw_gbps / (params * 0.5)) if bw_gbps else 0
    fits = vram_gb >= vram_need or not vram_gb
    tag  = ' <-- recommended' if fits and first_fit else ''
    if fits and first_fit: first_fit = False
    print(f'{i+1:<3} {mid:42} {params:>5.1f}B {vram_need:>8.1f} GB {"YES" if fits else "NO":>5} {tps:>5} t/s{tag}')

### 2 · Config — *the only cell you normally edit*
Set your **base model**, **dataset mode** (0 generate / 1 structure / 2 bring-your-own), **loop size**, and optional **LLM + export** settings. Every cell below reads its values from here.

In [ ]:
from pathlib import Path
import re as _re

# ── Smart path defaults ────────────────────────────────────────────────
def _base_dir():
    if IN_COLAB: return Path('/content')
    try:    return Path.cwd()
    except: return Path.home()

_base        = _base_dir()
_default_run = str(_base / 'touster_run')
_default_out = str(_base / 'touster_out')

# ═════════════════════════════ BASE MODEL ═════════════════════════════
BASE_MODEL  = 'unsloth/Qwen2.5-0.5B-Instruct'  # pick from hardware table above

# ══════════════════════════════ DATASET ═══════════════════════════════
# Base defaults — the mode block you uncomment below overrides what it needs.
DATASET_TOPIC  = ''       # topic string (mode 0)
DATASET_PATH   = None     # file / URL / HF id (modes 1, 2)
NUM_SAMPLES    = 200      # samples to generate/keep (modes 0, 1)
GEN_BATCH_SIZE = 10       # Q&A pairs per LLM call (mode 0); lower if JSON errors

# Uncomment ONE mode block:

# ═══════════════════════════════ MODE-0 ═══════════════════════════════
# Generate Q&A pairs from scratch using an LLM
DATASET_MODE   = 0
DATASET_TOPIC  = 'Python coding tips and tricks'

# ═══════════════════════════════ MODE-1 ═══════════════════════════════
# Structure your own raw text into Q&A pairs (needs LLM)
# DATASET_MODE  = 1
# DATASET_PATH  = 'my_raw_text.txt'

# ═══════════════════════════════ MODE-2 ═══════════════════════════════
# Load/download an existing dataset (no LLM). DATASET_PATH accepts:
#   - Local file  :  'my_dataset.jsonl'
#   - Direct URL  :  'https://example.com/data.jsonl'
#   - HF dataset  :  'tatsu-lab/alpaca'  or  'org/name/train' (with split)
# DATASET_MODE  = 2
# DATASET_PATH  = ''

# ═══════════════════════════ TRAINING LOOP ════════════════════════════
MAX_TRIALS  = 20    # trials  (CPU: 3-5; GPU: up to 20)
TRIAL_STEPS = 50    # steps per trial

# ═══════════════════════ LLM CLIENT (optional) ════════════════════════
API_KEY      = ''
API_BASE     = ''
API_MODEL    = ''
OLLAMA_PORT  = 11434
OLLAMA_MODEL = 'qwen2.5:3b'

# ═══════════════════════ POST-TRAINING & EXPORT ═══════════════════════
SAVE_LOCAL     = True
LOCAL_SAVE_DIR = _default_out
EXPORT_MERGED  = True
EXPORT_GGUF    = True
GGUF_QUANTIZE  = 'q4_k_m'

# ════════════════════════ HF HUB (optional) ═══════════════════════════
HF_TOKEN   = ''
HF_REPO_ID = ''

# ═══════════════════════════ RUN DIRECTORY ════════════════════════════
RUN_DIR = _default_run

# ──────────────────────────────────────────────────────────────────────
run_dir        = Path(RUN_DIR)
local_save_dir = Path(LOCAL_SAVE_DIR)
run_dir.mkdir(parents=True, exist_ok=True)
if SAVE_LOCAL:
    local_save_dir.mkdir(parents=True, exist_ok=True)

# Pre-declare so all later cells can reference these names
validated_path = None
llm_client     = None
recipe         = None   # set in cell 05

def _source_type(s):
    """Return 'url', 'hf', or 'local' for a dataset source string."""
    if not s: return 'local'
    if s.startswith('http://') or s.startswith('https://'): return 'url'
    if _re.match(r'^[A-Za-z0-9_\-\.]+/[A-Za-z0-9_\-\.]+(/[A-Za-z0-9_\-\.]+)?$', s):
        return 'hf'
    return 'local'

if DATASET_MODE == 2:
    if not DATASET_PATH:
        raise ValueError('Mode 2: set DATASET_PATH to a local file, URL, or HuggingFace dataset ID.')
    _stype = _source_type(str(DATASET_PATH))
    if _stype == 'local':
        validated_path = Path(DATASET_PATH)
        if not validated_path.exists():
            raise FileNotFoundError(f'Dataset not found: {validated_path.resolve()}')
        print(f'Mode 2 — local file: {validated_path}  (cell 04 will be skipped)')
    elif _stype == 'url':
        print(f'Mode 2 — URL: {DATASET_PATH}  (will download in cell 04)')
    else:
        print(f'Mode 2 — HuggingFace: {DATASET_PATH}  (will fetch in cell 04)')
elif DATASET_MODE == 0 and not DATASET_TOPIC:
    raise ValueError('Mode 0: set DATASET_TOPIC to a non-empty topic string.')
elif DATASET_MODE == 1 and not DATASET_PATH:
    raise ValueError('Mode 1: set DATASET_PATH to your raw .txt/.md file.')
else:
    if DATASET_MODE in (0, 1) and not API_KEY and not OLLAMA_MODEL:
        print('WARNING: mode 0/1 needs an LLM — set API_KEY or OLLAMA_MODEL')

_llm_label = ('API — ' + (API_BASE or 'openai.com') + ' / ' + (API_MODEL or 'default')) if API_KEY \
    else ('Ollama — ' + (OLLAMA_MODEL or 'auto')) if (OLLAMA_MODEL or DATASET_MODE in (0,1)) \
    else 'none (heuristic)'

print(f'Environment : {env_name}  backend={PLATFORM}')
print(f'Model       : {BASE_MODEL}')
if DATASET_MODE == 2:
    print(f'Dataset     : mode 2  source={DATASET_PATH}')
else:
    print(f'Dataset     : mode {DATASET_MODE}  topic="{DATASET_TOPIC}"  n={NUM_SAMPLES}  batch={GEN_BATCH_SIZE}')
print(f'Loop        : {MAX_TRIALS} trials × {TRIAL_STEPS} steps')
print(f'LLM client  : {_llm_label}')
print(f'Run dir     : {run_dir}')


### 3 · Install
Installs Touster for your detected backend (GPU / CPU / Apple-MLX). One-time, ~1–2 min. Nothing to change.

In [4]:
import subprocess, sys, os
from pathlib import Path

# Kernel's cwd may have been deleted — anchor everything to home dir
try:
    os.getcwd()
except OSError:
    os.chdir(Path.home())

_safe_cwd = str(Path.home())

def run(cmd, check=True):
    return subprocess.run(cmd, shell=True, check=check, cwd=_safe_cwd)

_TOUSTER_PKG = 'touster'
_TOUSTER_URL = 'git+https://github.com/Patan-Sameer66/touster.git'

if PLATFORM == 'cuda' and (IN_COLAB or ON_LINUX):
    r = run('pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q', check=False)
    if r.returncode != 0:
        print('Unsloth install failed — continuing with HF+PEFT backend')
    run(f'pip install "{_TOUSTER_PKG}[gpu] @ {_TOUSTER_URL}" -q')
    print('Installed: Touster[gpu]')

elif PLATFORM == 'cuda' and ON_WINDOWS:
    run(f'pip install "{_TOUSTER_PKG}[gpu] @ {_TOUSTER_URL}" -q')
    print('Installed: Touster[gpu]  (HF+PEFT — Unsloth not supported on Windows)')

elif PLATFORM == 'mlx':
    run(f'pip install "{_TOUSTER_PKG}[mlx] @ {_TOUSTER_URL}" -q')
    print('Installed: Touster[mlx]')

else:
    run(f'pip install "{_TOUSTER_PKG}[cpu] @ {_TOUSTER_URL}" -q')
    print('Installed: Touster[cpu]  (CPU-only)')

### 4 · Build dataset
Runs your chosen mode — **generate** (0), **structure raw text** (1), or **load/download** (2) — then dedups and validates. Auto-skipped when a local mode-2 file was already resolved in Cell 2.

In [5]:
from touster.config import DatasetConfig
from touster.dataset.modes import run_dataset_mode
from touster.llm.factory import build_client

# Mode 2 + local file: already resolved in cell 02, nothing to do
if DATASET_MODE == 2 and validated_path is not None:
    print(f'Skipping generation (mode 2, local file) — using: {validated_path}')
else:
    # Mode 2 + URL/HF: download/convert; mode 0/1: generate with LLM
    if DATASET_MODE in (0, 1):
        if API_KEY or API_BASE:
            llm_client = build_client(api_key=API_KEY, api_base=API_BASE, model=API_MODEL)
            print(f'LLM: OpenAI-compatible ({API_BASE or "api.openai.com"}) model={API_MODEL or "default"}')
        else:
            try:
                llm_client = build_client(ollama_port=OLLAMA_PORT, model=OLLAMA_MODEL)
                available = llm_client.list_models()
                picked = OLLAMA_MODEL or (available[0] if available else '')
                print(f'LLM: Ollama  model={picked or "?"}  available={available}')
            except Exception as e:
                print(f'No LLM ({e}) — heuristic-only loop')

    _gen_model = OLLAMA_MODEL if not API_KEY else API_MODEL

    ds_cfg = DatasetConfig(
        mode=DATASET_MODE,
        num_samples=NUM_SAMPLES,
        gen_batch_size=GEN_BATCH_SIZE,
        model=_gen_model,
        dataset_path=Path(DATASET_PATH) if DATASET_PATH else None,
        prompt=DATASET_TOPIC if DATASET_MODE == 0 else '',
    )

    validated_path = run_dataset_mode(ds_cfg, run_dir, llm_client)
    print(f'Dataset: {validated_path}')


### 5 · Preview
Shows a few samples, token-count stats, and the starting hyperparameters — a quick sanity check before training.

In [4]:
from touster.config import RecipeConfig
from touster.dataset.preview import print_preview

recipe = RecipeConfig(base_model=BASE_MODEL)
print_preview(validated_path, recipe)


### 6 · Self-improvement loop  
Runs short trials, proposing a new recipe each time (always **branching from the best trial so far**), keeps the lowest-loss one, then trains that winning recipe to completion. Outputs the final adapter.

In [5]:
from touster.config import LoopConfig
from touster.tuning.loop import run_loop

loop_cfg = LoopConfig(
    max_trials=MAX_TRIALS,
    trial_max_steps=TRIAL_STEPS,
    judge_top_k=3,
    judge_prompts=20,
    use_llm_proposer=bool(llm_client),
)

best_recipe, adapter_path = run_loop(
    recipe=recipe,
    loop_cfg=loop_cfg,
    dataset_path=validated_path,
    run_dir=run_dir,
    client=llm_client,
)

print(f'Adapter : {adapter_path}')
print(f'Recipe  : lr={best_recipe.learning_rate:.2e} rank={best_recipe.lora_rank} scheduler={best_recipe.scheduler}')

### 7 · Trial log
Table of every trial's eval loss (bits-per-byte, lower = better) and which recipe changes were kept. Read-only.

In [ ]:
from touster.state import load_experiments, load_state

state = load_state(run_dir)
exps  = load_experiments(run_dir)

print(f'Trials: {len(exps)}   Best bpb: {state.best_bpb:.4f}  (trial {state.best_trial_id})')
print()
print(f'{"#":>5}  {"bpb":>8}  {"kept":>5}  recipe diff')
print('-' * 65)
for e in exps:
    print(f'{e.trial_id:>5}  {e.eval_bpb:>8.4f}  {"YES" if e.kept else "-":>5}  {e.recipe_diff}')

### 8 · Base vs fine-tuned
Generates answers from the original and the fine-tuned model side-by-side on fixed prompts, so you can see what training changed.

In [ ]:
from touster.dashboard.compare import ModelPair

pair = ModelPair(base_model_id=BASE_MODEL, adapter_path=adapter_path)
pair.load()

TEST_PROMPTS = [
    '<|im_start|>user\nExplain list comprehensions in Python.<|im_end|>\n<|im_start|>assistant\n',
    '<|im_start|>user\nWhat is a decorator?<|im_end|>\n<|im_start|>assistant\n',
]

for prompt in TEST_PROMPTS:
    q        = prompt.split('user\n')[1].split('<|im_end|>')[0].strip()
    base_out = pair.generate_base(prompt, max_new_tokens=150)
    ft_out   = pair.generate_finetuned(prompt, max_new_tokens=150)
    print('=' * 70)
    print(f'Q: {q}')
    print(f'\n[BASE]\n{base_out.strip()}')
    print(f'\n[FINE-TUNED]\n{ft_out.strip()}')
    print()

pair.unload()

### 9 · Try your own prompt
Edit `YOUR_PROMPT` and compare base vs fine-tuned on it interactively.

In [ ]:
from touster.dashboard.compare import ModelPair

YOUR_PROMPT = '<|im_start|>user\nHow do I use *args and **kwargs?<|im_end|>\n<|im_start|>assistant\n'
MAX_TOKENS  = 200

_pair = ModelPair(base_model_id=BASE_MODEL, adapter_path=adapter_path)
_pair.load()

print('BASE MODEL\n' + '-' * 50)
print(_pair.generate_base(YOUR_PROMPT, max_new_tokens=MAX_TOKENS).strip())
print('\nFINE-TUNED\n' + '-' * 50)
print(_pair.generate_finetuned(YOUR_PROMPT, max_new_tokens=MAX_TOKENS).strip())

_pair.unload()

### 10 · Export — merged weights
Merges the LoRA adapter into the base model → standalone **fp16 weights** for 🤗 transformers. Toggle with `EXPORT_MERGED` in Cell 2.

In [ ]:
from touster.export.merge import export_merged
import shutil

merged_path = None
if EXPORT_MERGED:
    merged_path = export_merged(adapter_path, run_dir, dtype='float16')
    if SAVE_LOCAL:
        dest = local_save_dir / 'merged_weights'
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(merged_path, dest)
        print(f'Merged (local): {dest}')
    else:
        print(f'Merged: {merged_path}')
else:
    print('Skipping merged export (EXPORT_MERGED=False)')

### 11 · Export — GGUF
Converts to quantized **GGUF** for Ollama / llama.cpp. Set the level with `GGUF_QUANTIZE` (`q4_k_m` / `q8_0` / `f16`) in Cell 2.

In [ ]:
from touster.export.gguf import export_gguf
import shutil

gguf_path = None
if EXPORT_GGUF:
    gguf_path = export_gguf(adapter_path, run_dir, quantization=GGUF_QUANTIZE)
    if SAVE_LOCAL:
        dest = local_save_dir / gguf_path.name
        shutil.copy2(gguf_path, dest)
        print(f'GGUF (local): {dest}')
    else:
        print(f'GGUF: {gguf_path}')
else:
    print('Skipping GGUF export (EXPORT_GGUF=False)')

### 12 · Model card
Writes a README describing the recipe, and — if `HF_TOKEN` + `HF_REPO_ID` are set in Cell 2 — pushes the model to the 🤗 Hub.

In [ ]:
from touster.export.modelcard import write_model_card

if HF_TOKEN and HF_REPO_ID:
    import huggingface_hub
    huggingface_hub.login(token=HF_TOKEN, add_to_git_credential=False)

card_path = write_model_card(
    recipe=best_recipe,
    run_dir=run_dir,
    push_to_hub=bool(HF_TOKEN and HF_REPO_ID),
    repo_id=HF_REPO_ID,
)
print(f'Model card: {card_path}')
if HF_TOKEN and HF_REPO_ID:
    print(f'Hub: https://huggingface.co/{HF_REPO_ID}')

### 13 · Summary
Recaps the best score, the winning recipe, and lists every saved artifact with its size.

In [ ]:
from pathlib import Path
from touster.state import load_state

state = load_state(run_dir)
print(f'Best bpb   : {state.best_bpb:.4f}  (trial {state.best_trial_id})')
print(f'Best recipe: lr={best_recipe.learning_rate:.2e} rank={best_recipe.lora_rank} alpha={best_recipe.lora_alpha}')
print()

artifacts = [
    ('adapter',        adapter_path,  True),
    ('merged (fp16)', merged_path,   EXPORT_MERGED),
    ('gguf',          gguf_path,     EXPORT_GGUF),
    ('model card',    card_path,     True),
]

print('Artifacts:')
for name, path, enabled in artifacts:
    if not enabled or path is None:
        print(f'  {name:<18} skipped')
        continue
    try:
        p = Path(path)
        mb = sum(f.stat().st_size for f in p.rglob('*') if f.is_file()) / 1024**2 if p.is_dir() else p.stat().st_size / 1024**2
        print(f'  {name:<18} {path}  ({mb:.0f} MB)')
    except Exception:
        print(f'  {name:<18} {path}')

if SAVE_LOCAL:
    print()
    print(f'Local copies: {local_save_dir}')
    for f in sorted(local_save_dir.rglob('*'))[:10]:
        print(f'  {f}')